# SnowGuard - Phase 1: RSCD -> MobileNetV2 road-surface classifier

Six classes: `dry, wet, water, fresh_snow, melted_snow, ice`. RSCD's 27 friction x material x unevenness labels are collapsed to the friction state.

**Recommended platform: Kaggle Notebook** (GPU P100/T4, Internet on) with the dataset `cristvollerei/rscd-dataset-1million` attached via *Add Input*. Colab also works but has to download ~15 GB.

Leakage control: RSCD's official `vali_20k` / `test_50k` are random samples of the same drives as `train`, so this notebook splits `train` by **capture session** (timestamp gaps) instead. The manifest of the split is saved for reproducibility.

Training checkpoints every epoch; if the session drops, re-run all cells with the same `OUTPUT_DIR` and it resumes.

In [ ]:
import os, sys, glob, subprocess

ON_KAGGLE = os.path.exists('/kaggle/input')
ON_COLAB = 'google.colab' in sys.modules

# Where the snowguard repo lives. Either set REPO_URL to your GitHub repo,
# or upload the `ml/` folder (Kaggle: as a dataset; Colab: to Drive) and set ML_DIR.
REPO_URL = ''
ML_DIR = None

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/snowguard'   # persists across disconnects
elif ON_KAGGLE:
    WORK = '/kaggle/working'
else:
    WORK = os.path.abspath('..')
os.makedirs(WORK, exist_ok=True)

def find_dir(root, match, max_depth=6):
    """Breadth-first directory search that never lists RSCD's huge image folders."""
    skip = {'train', 'test_50k', 'vali_20k', '.venv', 'runs', '__pycache__'}
    level = [root]
    for _ in range(max_depth):
        nxt = []
        for d in level:
            if match(d):
                return d
            if os.path.basename(d) in skip:
                continue
            try:
                nxt += [e.path for e in os.scandir(d) if e.is_dir()]
            except OSError:
                pass
        level = nxt
    return None

if REPO_URL:
    repo = os.path.join('/tmp' if ON_KAGGLE else WORK, 'snowguard')
    if not os.path.exists(repo):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo], check=True)
    ML_DIR = os.path.join(repo, 'ml')
elif ML_DIR is None:
    has_pkg = lambda d: os.path.isfile(os.path.join(d, 'snowguard_ml', '__init__.py'))
    ML_DIR = (find_dir('/kaggle/input', has_pkg) if ON_KAGGLE else None) or find_dir(WORK, has_pkg)
    assert ML_DIR, 'Set REPO_URL or ML_DIR so the snowguard_ml package can be found'

sys.path.insert(0, ML_DIR)
print('ML_DIR =', ML_DIR, '| WORK =', WORK)

In [ ]:
import tensorflow as tf, keras
print('TF', tf.__version__, '| Keras', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Locate the RSCD train directory.
DATA_ROOT = None
is_rscd_train = lambda d: os.path.basename(d) == 'train' and os.path.isdir(os.path.join(d, 'ice'))
if ON_KAGGLE:
    DATA_ROOT = find_dir('/kaggle/input', is_rscd_train)
elif ON_COLAB:
    # Needs ~/.kaggle/kaggle.json (Kaggle > Settings > Create New API Token).
    target = '/content/rscd'
    if not os.path.exists(target):
        from google.colab import files
        if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
            files.upload()
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            os.replace('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
            os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
        subprocess.run(['kaggle', 'datasets', 'download', '-d', 'cristvollerei/rscd-dataset-1million',
                        '-p', target, '--unzip'], check=True)
    DATA_ROOT = find_dir(target, is_rscd_train)

assert DATA_ROOT, 'RSCD train/ directory not found'
print('DATA_ROOT =', DATA_ROOT)
print(sorted(os.listdir(DATA_ROOT)))

In [ ]:
# Training config. 8000/class x 6 = 48k train images fits a Kaggle GPU session (~2-3 h).
OUTPUT_DIR = os.path.join(WORK, 'runs', 'phase1_mnv2')

args = [
    '--data-root', DATA_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--max-per-class', '8000',
    '--max-eval-per-class', '2000',
    '--batch-size', '64',
    '--head-epochs', '5',
    '--finetune-epochs', '15',
    '--finetune-layers', '60',
    '--ice-weight', '1.5',
    '--session-gap', '300',
]
if tf.config.list_physical_devices('GPU'):
    args.append('--mixed-precision')

In [ ]:
from snowguard_ml import train
train.main(args)

In [ ]:
import json
from IPython.display import Image, display
from snowguard_ml.evaluate import format_report

for tag in ('keras_val', 'keras_test'):
    p = os.path.join(OUTPUT_DIR, f'metrics_{tag}.json')
    if os.path.exists(p):
        print(format_report(json.load(open(p))), '\n')
display(Image(os.path.join(OUTPUT_DIR, 'confusion_matrix_keras_test.png')))
display(Image(os.path.join(OUTPUT_DIR, 'gradcam_test.png')))

In [ ]:
# Bundle the deliverables (model, labels, metrics, plots, split manifest) for download.
import shutil
keep = ['snowguard_mobilenetv2.keras', 'labels.json', 'manifest.csv', 'summary.json',
        'metrics_keras_val.json', 'metrics_keras_test.json',
        'confusion_matrix_keras_val.png', 'confusion_matrix_keras_test.png', 'gradcam_test.png']
bundle = os.path.join(WORK, 'phase1_artifacts')
os.makedirs(bundle, exist_ok=True)
for f in keep:
    src = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, bundle)
print(shutil.make_archive(bundle, 'zip', bundle))